In [1]:
using Revise
using TTNKit
using Test
using ITensors
using Plots
using LinearAlgebra
using LaTeXStrings
using KrylovKit
using Printf, BenchmarkTools, Random

In [2]:
create_wavefunction(sz::NTuple{D, Int}) where{D} = create_wavefunction(ComplexF64, sz)
create_wavefunction(elT, sz::NTuple{D,Int}) where{D} = normalize!(randn(elT, sz))

create_wavefunction (generic function with 2 methods)

In [49]:
function patron_application!(ttn::TreeTensorNetwork, wf_coefs::Array, op_ins::String; maxdim::Int = maxlinkdim(ttn), normalize::Bool = true)
    net = TTNKit.network(ttn)
    lat = TTNKit.physical_lattice(net)

    all(size(lat) .== size(wf_coefs)) || error("Trying to apply a patron wavefunction of dimensionality $(size(wf_coefs)) to a TTN defined on a lattice $(size(lat))")


    patron_mpo = wf_mpo(wf_coefs, lat, op_ins)

    for p in eachindex(TTNKit.lattice(net, 1))
        ttn = TTNKit.move_ortho!(ttn, (1,p))
        pc = TTNKit.child_nodes(net, (1, p))
        Tpat = map(j -> patron_mpo[j], last.(pc))

        pr = TTNKit.parent_node(net, (1,p))
        
        Tt  = ttn[(1, p)]

        rind = TTNKit.commonind(Tt, ttn[pr])
        linds = TTNKit.uniqueinds(Tt, rind)
        Tn = TTNKit.noprime(TTNKit.contract(Tt, Tpat...))



        A, R = TTNKit.factorize(Tn, linds, maxdim = maxdim, tags = TTNKit.tags(rind))
        
        ttn[(1,p)] = A
        ttn[pr] = ttn[pr] * R    
        
        # moving the orhto center
        ttn.ortho_center[1] = pr[1]
        ttn.ortho_center[2] = pr[2]
        # this has to be deleted in the future... dont need this anymore
        ttn.ortho_direction[1][p] = TTNKit.number_of_child_nodes(net, (1,p)) + 1
        ttn.ortho_direction[pr[1]][pr[2]] = -1
    end


    # now move to the higher layers layer by layer, excluding the top node

    for ll in 2:TTNKit.number_of_layers(net)-1
        for p in 1:TTNKit.number_of_tensors(net, ll)
            pp = (ll, p)
            ttnc = TTNKit.move_ortho!(ttn, pp)
            
            Tt = ttn[pp]

            pr = TTNKit.parent_node(net, pp)

            pc = TTNKit.child_nodes(net, pp)
            
            linds = map(p -> TTNKit.commonind(Tt, ttn[p]), pc)
            ptags = "Link,nl=$(ll),np=$(p)"#tags(commonind(Tt, ttn[pr]))
        
            A, R = factorize(Tt, linds; maxdim = maxdim, tags = ptags)
            ttn[pp] = A
            ttn[pr] = ttn[pr] * R
        
            # moving the orhto center
            ttn.ortho_center[1] = pr[1]
            ttn.ortho_center[2] = pr[2]
            # this has to be deleted in the future... dont need this anymore
            ttn.ortho_direction[ll][p] = TTNKit.number_of_child_nodes(net, (ll,p)) + 1
            ttn.ortho_direction[pr[1]][pr[2]] = -1
        end
    end

    if normalize
        tpnd = (TTNKit.number_of_layers(net), 1) 
        T = ttn[tpnd]
        ttn[tpnd] = T/norm(T)
    end

    return ttn
end

patron_application! (generic function with 2 methods)

In [50]:
function wf_mpo(wf, lat, op_ins)
    ampo = TTNKit.OpSum()

    for pci in keys(wf)
        pt = to_indices(wf, (pci,))

        plin = TTNKit.linear_ind(lat, pt)
        ampo += wf[pci], op_ins, plin
    end

    return TTNKit.MPO(ampo, TTNKit.siteinds(lat))
end

wf_mpo (generic function with 2 methods)

In [54]:
#=
begin n_layer = 4; nd = TTNKit.ITensorNode; conserve_qns = true; maxdim = 4; normalize = true

    #t = 1; phi = 0.15*2*π; Nbosons = 12; vpinning = 2.5
    net = TTNKit.BinaryRectangularNetwork(n_layer, nd, "Boson"; conserve_qns = conserve_qns)
    #net = TTNKit.BinaryChainNetwork(n_layer, nd, "Boson"; conserve_qns = conserve_qns)
    lat = TTNKit.physical_lattice(net)
    
    nsites = TTNKit.number_of_sites(net)
    
    
    states = fill("0", nsites) 
    
    ttn = TTNKit.ProductTreeTensorNetwork(net, states)
    ttn0 = deepcopy(ttn)
    
    wf_coefs = create_wavefunction(Float64, size(lat))
    #wf_coefs = randn(Float64, size(lat))

    ttn = patron_application!(ttn, wf_coefs, "Adag"; maxdim = maxdim, normalize = normalize)

    n_ex = real.(TTNKit.expect(ttn, "n"))
end
sum(n_ex)=#

In [13]:
#net = TTNKit.BinaryRectangularNetwork(4, nd, "Boson"; conserve_qns = conserve_qns);
#lat = TTNKit.physical_lattice(net);

In [52]:
#wf_mpo(create_wavefunction(Float64,(4,4)),lat,"Adag",5)

#let   wf_coefs.^2
#end